# The Determinants of Turnout

In [41]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [42]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [43]:
%%stata

import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date
gen day = date(date, "YMD")
format day %td
gen year = year(day)

* Encode token for clustering
encode gecko_id, gen(token)

* Independent variables
local complexity multi_choices weighted quorum delegation
local topic protocol_security treasury_expenditure user_incentive_increase tokenomics voting_mechanism_change


* Label variables
label var weighted "\${\it Weighted}_{i,t}\$"
label var quorum "\${\it Quorum}_{i,t}\$"
label var multi_choices "\${\it Multiple Choices}_{i,t}\$"
label var have_discussion "\${\it Discussion}_{i,t}\$"
label var delegation "\${\it Delegation}_{i,t}\$"
label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"
label var concensus_diff "\${\it \Delta Concensus}_{i,t}\$"

* Topic labels
label var protocol_security "\${\it Protocol Security}_{i,t}\$"
label var treasury_expenditure "\${\it Treasury Expenditure}_{i,t}\$"
label var user_incentive_increase "\${\it User Incentive Increase}_{i,t}\$"
label var tokenomics "\${\it Tokenomics}_{i,t}\$"
label var voting_mechanism_change "\${\it Voting Mechanism Change}_{i,t}\$"


. 
. import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(65 vars, 2,830 obs)

. 
. * Parse date
. gen day = date(date, "YMD")

. format day %td

. gen year = year(day)

. 
. * Encode token for clustering
. encode gecko_id, gen(token)

. 
. * Independent variables
. local complexity multi_choices weighted quorum delegation

. local topic protocol_security treasury_expenditure user_incentive_increase to
> kenomics voting_mechanism_change

. 
. 
. * Label variables
. label var weighted "\${\it Weighted}_{i,t}\$"

. label var quorum "\${\it Quorum}_{i,t}\$"

. label var multi_choices "\${\it Multiple Choices}_{i,t}\$"

. label var have_discussion "\${\it Discussion}_{i,t}\$"

. label var delegation "\${\it Delegation}_{i,t}\$"

. label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"

. label var concensus_diff "\${\it \Delta Concensus}_{i,t}\$"

. 
. * Topic labels
. label var protocol_security "\${\it Protocol Secu

## Proposal Characteristics

In [44]:
%%stata

******************************************************
* PANEL REGRESSIONS
******************************************************

eststo clear

    * Run baseline regression
    reghdfe non_whale_participation `complexity' have_discussion , absorb(year categories) 
    eststo baseline
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"


    * Run regression with discussion characteristics    
    reghdfe non_whale_participation `complexity' concensus_diff, absorb(year categories) 
    eststo discussion
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"


    * Run regression with user characteristics
    reghdfe non_whale_participation `complexity' have_discussion non_whale_user, absorb(year categories)
    eststo user
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"

* Export LaTeX table
esttab                                                           ///
    baseline discussion user                                     ///
    using "$TABLES/reg_participation.tex", replace               ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    order(delegation have_discussion concensus_diff non_whale_user multi_choices weighted quorum) ///
    mgroups("\${\it Participation}^{Small}_{i,t}\$",             ///
            span                                                 ///
            pattern(1 0 0)                                       ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-4} & \multicolumn{1}{c}{Full} & \multicolumn{1}{c}{Discussion \$\geq\$ 2} & \multicolumn{1}{c}{Have Dapp} \\ \cmidrule(lr){2-4} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} \\ \midrule) ///
            substitute("\_" "_")                                ///
    stats(fe_proposal fe_time N r2_a,                            ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Category FE" "Year FE" "Observations" "Adjusted R²"))


. 
. ******************************************************
. * PANEL REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
.     * Run baseline regression
.     reghdfe non_whale_participation `complexity' have_discussion , absorb(yea
> r categories) 
(dropped 2 singleton observations)
(MWFE estimator converged in 7 iterations)

HDFE Linear regression                            Number of obs   =      2,665
Absorbing 2 HDFE groups                           F(   5,   2646) =      52.28
                                                  Prob > F        =     0.0000
                                                  R-squared       =     0.3176
                                                  Adj R-squared   =     0.3129
                                                  Within R-sq.    =     0.0899
                                                  Root MSE        =     0.0373

----------------------------------------------------------------------------

## Topics

In [45]:
%%stata

******************************************************
* TOPIC REGRESSIONS
******************************************************

eststo clear

* Small participation
reghdfe non_whale_participation `topic', absorb(year categories) vce(cluster categories)
eststo small
estadd local fe_proposal "Y"
estadd local fe_time "Y"

* Export LaTeX table
esttab                                                           ///
    small                                                        ///
    using "$TABLES/reg_topic.tex", replace                       ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    noomitted eqlabels(none)                                     ///
    mgroups("\${\it Participation}^{Small}_{i,t}\$",             ///
            span                                                 ///
            pattern(1)                                           ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-2} & \multicolumn{1}{c}{(1)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                            ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Category FE" "Year FE" "Observations" "Adjusted R²"))


. 
. ******************************************************
. * TOPIC REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
. * Small participation
. reghdfe non_whale_participation `topic', absorb(year categories) vce(cluster 
> categories)
(dropped 2 singleton observations)
(MWFE estimator converged in 6 iterations)

HDFE Linear regression                            Number of obs   =      2,665
Absorbing 2 HDFE groups                           F(   5,      8) =     449.28
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.2820
                                                  Adj R-squared   =     0.2771
                                                  Within R-sq.    =     0.0424
Number of clusters (categories) =          9      Root MSE        =     0.0382

                             (Std. err. adjusted for 9 clusters in categories)
--------